### Training a Random Forest Regressor model
Training a random forest regressor that predicts FPL points for the upcoming GW for players. This is a general model, and uses position as one of the predictors. A future step might be to produce a separate model for each position so that position-specific features can be better considered.

Rolling game statistics are key to the model - they will be computed on the previous three games for each player, and used as predictor features.

In [1]:
import os
import torch
import numpy as np
import pandas as pd
from model import AdvancedLSTM
import pickle
from eval import season_performance_with_unlimited_transfers

In [2]:
base_path = os.getcwd()
base_path

'/Users/bragehs/Documents/FPL_forecast/backend/predictor'

In [3]:
data_path = os.path.join(base_path, 'processed_data')
data_path

'/Users/bragehs/Documents/FPL_forecast/backend/predictor/processed_data'

In [4]:
X_train = torch.load(data_path + '/X_train.pt', weights_only=True)
y_train = torch.load(data_path + '/y_train.pt', weights_only=True)
train_mapping = pd.read_csv(data_path + '/train_mapping.csv')

X_val = torch.load(data_path + '/X_val.pt', weights_only=True)
y_val = torch.load(data_path + '/y_val.pt', weights_only=True)

X_test = torch.load(data_path + '/X_test.pt', weights_only=True)
y_test = torch.load(data_path + '/y_test.pt', weights_only=True)
test_mapping = pd.read_csv(data_path + '/test_mapping.csv')

In [5]:
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")

X_train shape: torch.Size([543691, 5, 48])
y_train shape: torch.Size([543691, 1])
X_val shape: torch.Size([29725, 5, 48])
y_val shape: torch.Size([29725, 1])


In [9]:
best_model_data = torch.load("best_model.pth", map_location=torch.device('cpu'))

/var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/ipykernel_43474/2890380057.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_model_data = torch.load("best_model.pth"

In [10]:
best_model_data.keys()

dict_keys(['model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'epoch', 'best_performance', 'input_dim', 'hidden_dim', 'num_layers', 'num_fc_layers'])

In [11]:
model = AdvancedLSTM(
    hidden_dim=best_model_data['hidden_dim'],
    num_layers=best_model_data['num_layers'],
    input_dim= best_model_data['input_dim'],
    output_dim=1,
    num_fc_layers=best_model_data['num_fc_layers'],
) 
model.load_state_dict(best_model_data['model_state_dict'])

/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  warnings.warn(


<All keys matched successfully>

In [12]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

torch.Size([543691, 5, 48])
torch.Size([29725, 5, 48])
torch.Size([27283, 5, 48])


In [13]:
predictions = model(X_test).detach().numpy()
print(predictions.shape)
print(y_test.shape)

(27283, 1)
torch.Size([27283, 1])


In [14]:
test = pd.read_csv(data_path + '/test_data.csv')
test.columns

Index(['GW', 'last_1_assists', 'last_3_assists', 'last_5_assists',
       'last_all_assists', 'last_1_bonus', 'last_3_bonus', 'last_5_bonus',
       'last_all_bonus', 'last_1_bps', 'last_3_bps', 'last_5_bps',
       'last_all_bps', 'last_1_creativity', 'last_3_creativity',
       'last_5_creativity', 'last_all_creativity', 'last_1_clean_sheets',
       'last_3_clean_sheets', 'last_5_clean_sheets', 'last_all_clean_sheets',
       'last_1_goals_conceded', 'last_3_goals_conceded',
       'last_5_goals_conceded', 'last_all_goals_conceded',
       'last_1_goals_scored', 'last_3_goals_scored', 'last_5_goals_scored',
       'last_all_goals_scored', 'last_1_ict_index', 'last_3_ict_index',
       'last_5_ict_index', 'last_all_ict_index', 'last_1_influence',
       'last_3_influence', 'last_5_influence', 'last_all_influence',
       'last_1_minutes', 'last_3_minutes', 'last_5_minutes',
       'last_all_minutes', 'last_1_threat', 'last_3_threat', 'last_5_threat',
       'last_all_threat', 'last_1

In [15]:
remaining_lagged_features = pickle.load(open(data_path + '/remaining_lagged_features.pkl', 'rb'))

In [16]:
scores, total_score = season_performance_with_unlimited_transfers(
    y_test=y_test,
    predictions=predictions,
    remaining_lagged_features=remaining_lagged_features
)

/Users/bragehs/Documents/FPL_forecast/backend/predictor/eval.py:167: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  predictions_df_complete = pd.concat([gameweek_1, predictions_df], ignore_index=True)


Players with NaN total_points_last_season: []
Number of NaN values remaining: 0
0.0 :  lukasz_fabianski
1.0 :  sepp_van_den_berg
2.0 :  tyler_dibling
3.0 :  daniel_jebbison
Bench players: ['lukasz_fabianski', 'sepp_van_den_berg', 'tyler_dibling', 'daniel_jebbison']
Bench cost: 170
Simulating season with unlimited transfers for 38 gameweeks
Available budget per gameweek: 830

--- Gameweek 1.0 ---
Players available for GW 1.0: 668
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/61d1686abea94b74b03e9f483965f776-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/61d1686abea94b74b03e9f483965f776-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 30 COLUMNS
At line 4039 RHS
At line 4065 BOUNDS
At line 4734

In [17]:
total_score.item()

2371.0

In [18]:
scores

,team,gw_score
0,"[archie_gray, callum_bates, dominic_solanke_mi...",51.0
1,"[bernardo_veiga_de_carvalho_e_silva, bukayo_sa...",63.0
2,"[andrew_robertson, bukayo_saka, cristian_romer...",44.0
3,"[andrew_robertson, antoine_semenyo, dean_hende...",42.0
4,"[andre_onana, bukayo_saka, cole_palmer, danny_...",60.0
5,"[bryan_mbeumo, cole_palmer, cristian_romero, d...",102.0
6,"[antonee_robinson, brennan_johnson, bryan_mbeu...",69.0
7,"[bryan_mbeumo, cole_palmer, dwight_mcneil, erl...",59.0
8,"[brennan_johnson, cole_palmer, dwight_mcneil, ...",59.0
9,"[aaron_ramsdale, brennan_johnson, cole_palmer,...",57.0
